# Extracting roles, skills and requirements

This notebook turns the 1,068 data-role ads from `02_cleaning` into structured information:

1. **Role and seniority** from the job title
2. **Skills** from the full ad text, with a validation of the matches
3. **Required vs. nice-to-have** skills, from the wording around each skill
4. **Entry barriers:** experience, degree and German level

The results feed the analysis in `04_analysis` and the web app.

In [ ]:
# Load the data

from pathlib import Path
import html
import re

import pandas as pd

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
jobs = pd.read_pickle(PROCESSED_DIR / "jobs_data_roles.pkl")

print(jobs.shape)
print(jobs.columns.tolist())


# Clean the ad text for extraction

First a function *`clean_text`* is defined to clean the ad text by decoding HTML entities, removing Markdown markers, and keeping line breaks.

In [ ]:
def clean_text(s):
    """Readable ad text: decode HTML entities, remove Markdown markers, keep line breaks."""
    return (s.map(html.unescape)
             .str.replace(r"\\([*_])", r"\1", regex=True)               # unescape \* and \_
             .str.replace(r"\*\*|__", "", regex=True)                   # bold markers
             .str.replace(r"(?m)^[ \t]*#+[ \t]*", "", regex=True)       # heading markers
             .str.replace(r"(?m)^[ \t]*[*•][ \t]+", "- ", regex=True)   # bullets -> "- "
             .str.replace(r"[ \t]+", " ", regex=True)                   # collapse spaces
             .str.replace(r"\n\s*\n+", "\n", regex=True)                # drop empty lines
             .str.strip())

jobs["text"] = clean_text(jobs["stellenangebotsBeschreibung"])

print(jobs["text"].str.len().describe().round())
print("\n--- example ---\n")
print(jobs["text"].iloc[0][:800])

**The cleaned texts are readable and keep their structure.** HTML entities and Markdown markers are removed, while line breaks and bullet points stay. That matters for the next steps, because the example already shows the two things the extraction relies on:

- **Section headings** separate tasks ("Ihre Aufgaben bei uns") from requirements ("Was wir von Ihnen erwarten").
- **Hedging words mark optional requirements:** "Berufserfahrung in SAP, *idealerweise* mit SAP Business One".

Skills appear in both sections: here, Power BI, SQL, Power Query and DAX are already named in the tasks. So skills are searched in the whole text, not only in the requirements.

**The ads are long:** the median text has 3,569 characters, and the longest 9,908. The shortest has only 247 characters, so a few ads may contain too little text to extract much from.

### Ads that matched only through a company name

**Six ads were in the dataset only because the employer is called "NTT DATA".** The word "DATA" in the company prefix ("NTT DATA Deutschland SE: …") was enough to pass the data-role filter, although the job titles themselves contain no data term, e.g. "Client Partner Banking DACH", "Consultant DevSecOps/Application Security" or "Linux Administrator:in im Storage".

The cell strips the company prefix and checks the rest of the title with the same data terms as the final filter. Ads without a data term are removed, which leaves 11 of the 17 NTT DATA ads and 1,062 ads in total.

*Future work:* the prefix removal belongs in the filter in `02_cleaning`, so that the cleaning notebook itself is correct.

In [ ]:
# Ads that matched the data-role filter only through the company name "NTT DATA"
DATA_TITLE_V3 = (r"data|daten(?!schutz)|analy|statisti|intelligen"
                 r"|\bbi\b|\bsap bw|datasphere|snowflake"
                 r"|machine.?learning|maschinell|deep.?learning|computer.?vision"
                 r"|\bml|mlops|\bki\b|\bai\b|genai|\bllm|\bnlp\b|prompt.?engineer|\betl\b"
                 r"|applied.?scien|decision.?scien")   # same data terms as the final rule in 02_cleaning

no_company = jobs["stellenangebotsTitel"].str.replace(r"^NTT DATA[^:]*:\s*", "", case=False, regex=True)
company_only = (jobs["stellenangebotsTitel"].str.contains("NTT DATA", case=False)
                & ~no_company.str.contains(DATA_TITLE_V3, case=False, regex=True))

print(company_only.sum(), "ads removed:")
print(jobs.loc[company_only, "stellenangebotsTitel"].to_string())

jobs = jobs[~company_only].reset_index(drop=True)
print("\n", jobs.shape)

### Role from the title

**Each ad gets one role from its title.** The rules are checked in a fixed order and the first match wins: named roles first (Data Scientist, Data Engineer, ML / AI Engineer, BI / Analytics, Data Analyst), then two generic rules. A title with a data word *and* an engineering word ("Data Software Engineer", "Data Architekt") counts as Data Engineer; a title with only a generic AI term ("AI Engineer", "Künstliche Intelligenz") counts as ML / AI Engineer. The company prefix "NTT DATA" is removed before matching.

| Role | Ads |
|---|---|
| Data Engineer | 281 |
| Data Analyst | 247 |
| ML / AI Engineer | 211 |
| BI / Analytics | 119 |
| Data Scientist | 119 |
| Other data role | 85 |

**The rules were revised three times, each time based on a check of the output:**

- **Generic AI terms came too early:** "Data Analyst & AI Expert" was assigned to ML / AI. They now come last.
- **Data warehousing pulled BI jobs into Data Engineer:** "BI Developer / Data Warehouse Developer". Data warehouse and SAP Datasphere now count as BI.
- **Free-form data engineering titles fell into *Other*:** "Data Migration Engineer", "Data Architekt". The generic data-engineering rule catches 41 of them.

**Three checks support the result:**

- **Order matters for few ads:** 773 titles match exactly one role, 172 match two or three, and 117 match none (→ *Other*).
- **The official occupation broadly agrees:** 110 of the ads with the occupation *Data Engineer* are assigned *Data Engineer*, and 62 of the *Data-Analyst/in* ads *Data Analyst*. The occupation field is unreliable for single ads (see `02_cleaning`), so this is a plausibility check, not a validation.
- **The remaining *Other* roles are data management, governance, quality and strategy,** e.g. "Datenmanager", "Data Governance Manager", "Principal Data Strategist", which is what the category is for.

There are no hand labels for the roles, so their accuracy is not measured.

In [ ]:
# Company prefix removed before matching: "NTT DATA Deutschland SE:" contains "DATA"
# and would otherwise trigger the data rules (CHANGED)
title = jobs["stellenangebotsTitel"].str.replace(r"^NTT DATA[^:]*:\s*", "", case=False, regex=True)

# Role from the title. First match wins: named roles first, generic rules only at the end.
ROLES = {
    "Data Scientist":   r"data.?scien|datenwissenschaft|applied.?scien|decision.?scien",
    "Data Engineer":    (r"data.?engineer|dateningenieur|analytics.?engineer|\betl\b|data.?platform|snowflake"
                         r"|data.?integration|data.?architect|dataarchitekt|datenarchitekt|databricks|data.?pipeline"),
    "ML / AI Engineer": r"machine.?learning|maschinell|\bml\b|mlops|deep.?learning|computer.?vision|\bllm|\bnlp\b",
    "BI / Analytics":   (r"\bbi\b|business.?intelligen|power.?bi|\bsap bw|data.?warehouse|datasphere"
                         r"|sap analytics|\bsac\b|tableau|qlik|microstrategy|looker"),
    # data mining added: two data-mining ads were left in "Other" (CHANGED)
    "Data Analyst":     r"analy|statisti|data.?mining",
    # (?=...) is a lookahead: two together mean "contains a data word AND an engineering word, in any order"
    "Data Engineer (generic)":       r"^(?=.*(?:\bdata|\bdaten))(?=.*(?:engineer|archite[ck]t|entwickler|developer))",
    "ML / AI Engineer (generic AI)": r"\bai\b|\bki\b|genai|prompt.?engineer|künstliche.?intelligenz|artificial.?intelligence",
}

matches = pd.DataFrame({role: title.str.contains(p, case=False, regex=True) for role, p in ROLES.items()})
named = ["Data Scientist", "Data Engineer", "ML / AI Engineer", "BI / Analytics", "Data Analyst"]

jobs["role"] = (matches.idxmax(axis=1)
                .where(matches.any(axis=1), "Other data role")
                .replace({"ML / AI Engineer (generic AI)": "ML / AI Engineer",
                          "Data Engineer (generic)": "Data Engineer"}))

print(jobs["role"].value_counts(), "\n")

# Check 1: titles matching more than one role (only the order decides)
n_matches = matches[named + ["ML / AI Engineer (generic AI)"]].sum(axis=1)
print("roles matched per title:")
print(n_matches.value_counts().sort_index(), "\n")

multi = jobs.loc[n_matches > 1, ["stellenangebotsTitel", "role"]].copy()
multi["also matches"] = matches[n_matches > 1].apply(lambda r: ", ".join(r.index[r]), axis=1)
print(multi.sample(min(20, len(multi)), random_state=1).to_string(), "\n")

# Check 2: role vs. official occupation
top_occ = jobs["hauptberuf"].value_counts().head(8).index
sub = jobs[jobs["hauptberuf"].isin(top_occ)]
print(pd.crosstab(sub["hauptberuf"], sub["role"]), "\n")

# Check 3: what is left in "Other" for ads whose occupation is Data Engineer or Data Scientist
left = jobs[(jobs["role"] == "Other data role") & jobs["hauptberuf"].isin(["Data Engineer", "Data Scientist"])]
print(len(left), "ads in 'Other' with occupation Data Engineer / Data Scientist:")
print(left["stellenangebotsTitel"].to_string(), "\n")

# Check 4: ads decided by the generic data-engineering rule
by_generic = matches["Data Engineer (generic)"] & ~matches[named].any(axis=1)
print(by_generic.sum(), "ads assigned by the generic data-engineering rule")
print((by_generic & matches["ML / AI Engineer (generic AI)"]).sum(), "of them also contain a generic AI term\n")
print(jobs.loc[by_generic, "stellenangebotsTitel"].to_string(), "\n")

# Check 5: NTT DATA ads, to see whether any is in the dataset only because of the company name (NEW)
ntt = jobs["stellenangebotsTitel"].str.contains("NTT DATA", case=False)
print(ntt.sum(), "NTT DATA ads:")
print(jobs.loc[ntt, ["stellenangebotsTitel", "role"]].to_string())

In [ ]:
# "expert" removed: it also matched company names, e.g. "(expert …)" at the end of a title
SENIOR = r"\bsenior\b|\bsr\.?\b|\blead\b|principal|\bstaff\b|head of|leiter"
JUNIOR = r"\bjunior\b|\bjr\.?\b|einsteiger|trainee|graduate|absolvent|entry"

is_senior = title.str.contains(SENIOR, case=False, regex=True)
is_junior = title.str.contains(JUNIOR, case=False, regex=True)

jobs["seniority"] = "not stated"
jobs.loc[is_senior & ~is_junior, "seniority"] = "senior"
jobs.loc[is_junior & ~is_senior, "seniority"] = "junior"
jobs.loc[is_junior & is_senior, "seniority"] = "junior and senior"   # e.g. "(Junior/Senior) Data Analyst"

print(jobs["seniority"].value_counts(), "\n")
print(pd.crosstab(jobs["role"], jobs["seniority"], margins=True), "\n")

# Check 1: levels in parentheses, e.g. "(Senior) Data Scientist" = also open to non-seniors
in_parens = title.str.contains(r"\(\s*(?:senior|sr\.?|junior|jr\.?)\s*\)", case=False, regex=True)
print(in_parens.sum(), "titles with the level in parentheses:")
print(jobs.loc[in_parens, "seniority"].value_counts(), "\n")

# Check 2: 10 sample titles per seniority level
for level in jobs["seniority"].value_counts().index:
    print(f"--- {level}")
    sample = jobs.loc[jobs["seniority"] == level, "stellenangebotsTitel"]
    print(sample.sample(min(10, len(sample)), random_state=1).to_string(), "\n")

### Seniority from the title

**Most titles do not state a level at all: 757 of 1,062 ads.** 

250 titles are marked as senior (senior, lead, principal, staff, head of, Leiter) and only 55 as junior (junior, trainee, graduate, Absolvent, Einsteiger, which also matches "Quereinsteiger").

| Role | Junior | Not stated | Senior |
|---|---|---|---|
| BI / Analytics | 5 | 97 | 17 |
| Data Analyst | 25 | 180 | 42 |
| Data Engineer | 7 | 181 | 93 |
| Data Scientist | 7 | 86 | 26 |
| ML / AI Engineer | 8 | 149 | 54 |
| Other data role | 3 | 64 | 18 |

Data Engineer has the most senior titles (93), Data Analyst the most junior titles (25).

**Two limitations:**

- **64 titles give the level in parentheses**, e.g. "(Senior) Data Scientist" (44 senior, 20 junior). This usually means "also open to this level", so these ads are counted as senior or junior although they may accept others.
- **"Expert" was removed from the senior terms,** because it also matched company names at the end of titles.

Because so few titles state a level, the years of experience required in the ad text will be the more informative measure of how open ads are to newcomers.

## Skills

**Skills are found with a dictionary of 46 skills and their German and English variants.** Each skill is a regular expression searched in the whole ad text, because skills appear in the task section as well as in the requirements.

Three design choices handle the difficult cases:

- **German compounds:** most patterns have no word boundary at the end, so "Python" also matches "Pythonkenntnisse" and "SQL" matches "SQL-Abfragen".
- **Short or ambiguous names** get extra conditions: "R" is matched case-sensitively as a separate word and not before "&" (R&D); "Excel" not in "Excellence"; "Spark" not in "Sparkasse"; "Java" not in "JavaScript"; "DAX" not as the stock index ("DAX-Konzern").
- **SQL dialects count as SQL** (MySQL, PostgreSQL), but NoSQL does not.

The matches are checked by reading each hit in context (next cell).

In [ ]:
# Skill dictionary. Regex symbols used below:
#   (?i)    case-insensitive (only R and SAS are case-sensitive on purpose)
#   \b      word boundary: start or end of a word (space, hyphen, punctuation)
#           -> no \b at the end lets compounds match: "python" also finds "Pythonkenntnisse"
#   .?      one optional character of any kind: "power.?bi" = "Power BI", "Power-BI", "PowerBI"
#   a|b     a or b
#   (?:…)   group alternatives without capturing them
#   (?!…)   negative lookahead: NOT followed by …
#   (?<!…)  negative lookbehind: NOT preceded by …
#   \w*     any number of letters;  \s+ one or more spaces;  \d a digit
#   \+      a literal "+" (a plain + would mean "one or more")
SKILLS = {
    # programming languages
    "Python":                  r"(?i)python",                     # anywhere, incl. "Pythonkenntnisse"
    "SQL":                     r"(?i)(?<!no)sql",                 # SQL, MySQL, PostgreSQL, "SQL-Abfragen"; not NoSQL
    "R":                       r"(?<!d\.)(?<!d\. )\bR\b(?!\s*[&+])",   # not "R&D", "R+V", "i.d.R.", "i. d. R."
    "Java":                    r"(?i)\bjava(?!script)",           # word starting with "java", not JavaScript
    "JavaScript / TypeScript": r"(?i)javascript|typescript",      # either language
    "Scala":                   r"(?i)\bscala\b",                  # whole word only; not "scalable"
    "C++":                     r"(?i)c\+\+",                      # "C++" (the + signs escaped)
    "C#":                      r"(?i)\bc#",                       # "C#" at the start of a word
    "SAS":                     r"\bSAS\b",                        # capital SAS as a separate word
    "MATLAB":                  r"(?i)matlab",                     # anywhere
    "VBA":                     r"(?i)\bvba\b",                    # whole word
    # data engineering
    "Spark":                   r"(?i)\b(?:py)?spark\b",           # "Spark" or "PySpark" as a whole word; not "Sparkasse"
    "Kafka":                   r"(?i)kafka",                      # anywhere
    "Airflow":                 r"(?i)airflow",                    # anywhere
    "dbt":                     r"(?i)\bdbt\b",                    # whole word (the tool "dbt")
    "ETL / ELT":               r"(?i)\betl\b|\belt\b",            # the abbreviations as whole words
    "Data warehouse":          r"(?i)data.?warehous|\bdwh\b",     # "Data Warehouse", "Data-Warehousing", or "DWH"
    "Data modelling":          r"(?i)datenmodell|data.?modell?ing",  # "Datenmodellierung", "data modeling/modelling"
    "Snowflake":               r"(?i)snowflake",                  # anywhere
    "Databricks":              r"(?i)databricks",                 # anywhere
    "NoSQL":                   r"(?i)nosql|mongodb|cassandra",    # NoSQL or two common NoSQL databases
    # cloud and DevOps
    "AWS":                     r"(?i)\baws\b|amazon web services",  # abbreviation or full name
    "Azure":                   r"(?i)\bazure",                    # word starting with "Azure", e.g. "Azure-Cloud"
    "Google Cloud":            r"(?i)\bgcp\b|google cloud",       # abbreviation or full name
    "Docker":                  r"(?i)docker",                     # anywhere
    "Kubernetes":              r"(?i)kubernetes|\bk8s\b",         # full name or the short form "k8s"
    "Terraform":               r"(?i)terraform",                  # anywhere
    "Git":                     r"(?i)\bgit\b|github|gitlab",      # "Git" as a whole word, or GitHub/GitLab; not "digital"
    "Linux":                   r"(?i)linux",                      # anywhere
    # BI and reporting
    "Power BI":                r"(?i)power.?bi",                  # "Power BI", "Power-BI", "PowerBI"
    "Tableau":                 r"(?i)tableau",                    # anywhere
    "Qlik":                    r"(?i)qlik",                       # anywhere, incl. "QlikView", "Qlik Sense"
    "Excel":                   r"(?i)\bexcel(?!len)",             # "Excel", "Excelkenntnisse"; not "Excellence", "excellent"
    "DAX / Power Query":       r"(?i)power.?query|\bdax\b(?![- ]?(?:konzern|unternehmen|notiert|\d))",
                                                                  # Power Query, or "DAX" but not the stock index
                                                                  # ("DAX-Konzern", "DAX-Unternehmen", "DAX 40")
    "SAP":                     r"(?i)\bsap\b",                    # whole word, incl. "SAP-Kenntnisse"
    # machine learning and AI
    "Machine learning":        r"(?i)machine.?learning|maschinelle\w*\s+lernen|\bml\b",
                                                                  # "machine learning", "maschinelles Lernen", or "ML"
    "Deep learning":           r"(?i)deep.?learning|neuronale|neural net",  # incl. "neuronale Netze", "neural networks"
    "NLP":                     r"(?i)\bnlp\b|natural language|sprachverarbeitung",  # abbreviation, English or German term
    "Computer vision":         r"(?i)computer.?vision|bildverarbeitung",           # English or German term
    "LLM / GenAI":             r"(?i)\bllms?\b|large language model|generative (?:ai|ki)|\bgenai\b|\bgpt",
                                                                  # LLM(s), "generative AI/KI", GenAI, words starting with GPT
    "scikit-learn":            r"(?i)scikit|sklearn",             # full name or import name
    "TensorFlow":              r"(?i)tensorflow",                 # anywhere
    "PyTorch":                 r"(?i)pytorch",                    # anywhere
    "pandas":                  r"(?i)\bpandas\b",                 # whole word
    "Statistics":              r"(?i)statisti",                   # "Statistik", "statistical", "Statistiker"
    # ways of working
    "Agile / Scrum":           (r"(?i)\bscrum|\bkanban"
                                r"|\bagil\w*\s+(?:method|projekt|vorgehen|entwicklung|development|framework|practice|praktik)"),
                                                                  # Scrum, Kanban, or "agile Methoden/Projekte/Entwicklung";
                                                                  # "arbeit" removed: it matched "agile Arbeitskultur"
} 

In [ ]:
# Extract skills from job ads and summarize their occurrence
text = jobs["text"]
S = pd.DataFrame(data={skill: text.str.contains(p, regex=True) for skill, p in SKILLS.items()}) # boolean DataFrame indicating presence of each skill in each job ad
jobs["skills"] = S.apply(func=lambda row: list(row.index[row]), axis=1) # list of skills found in each job ad; .apply is used to iterate over each row

counts = S.sum().sort_values(ascending=False) # number of ads containing each skill
print(len(SKILLS), "skills in the dictionary\n")
print(pd.DataFrame({"ads": counts, "share": (counts / len(jobs)).round(3)}).to_string(), "\n")

per_ad = S.sum(axis=1) # number of skills found in each job ad; axis=1: summing across columns for each row
print("skills found per ad:")
print(per_ad.describe().round(decimals=1)) 
print((per_ad == 0).sum(), "ads with no skill found")

In [ ]:
def context(t, pattern, width=60):
    """Return the first match of a pattern with some text around it.

    Args:
        t (str): the text to search in
        pattern (str): the regex pattern to search for
        width (int): number of characters to include on each side of the match

    Returns:
        str: the match with its surrounding text, on one line
    """
    m = re.search(pattern, t)
    if m is None:   # cannot happen for texts selected by S, but keeps the function safe on its own
        return "…no match found…"
    # m.start() and m.end() are the positions of the match; max() stops the slice at the start of the text

    # the match itself is wrapped in >>> <<< so it stands out when reading by a human 
    start = max(0, m.start() - width)
    return ("…" + t[start:m.start()] + " >>>" + m.group() + "<<< " + t[m.end():m.end() + width] + "…").replace("\n", " ")


# Check the 15 most frequent skills plus all skills with risky patterns
RISKY = ["R", "SAS", "Excel", "Spark", "DAX / Power Query", "Git", "Scala", "C#",
         "Machine learning", "SAP", "Statistics", "Agile / Scrum", "ETL / ELT"]
to_check = list(dict.fromkeys(list(counts.index[:15]) + RISKY))   # dict.fromkeys removes duplicates, keeps the order

for skill in to_check:
    hits = jobs.loc[S[skill], "text"] # select the text of job ads where the skill is present
    print(f"=== {skill} ({len(hits)} ads)")
    
    for t in hits.sample(min(5, len(hits)), random_state=1): # sample up to 5 job ads containing the skill
        print("   ", context(t, SKILLS[skill])) # SKILLS[skill] is the regex pattern for the skill
    print()
    


In [ ]:
# Manual spot check: correct hits out of 5 random samples per skill (final patterns)
# Rule: correct = the word refers to a tool or method the job uses or requires
#       (incl. fields of study); wrong = company name, application instruction, employer description

spot_check = {
    "Python": 5, "SQL": 5, "Machine learning": 5, "Data modelling": 5,
    "Power BI": 5, "Azure": 5, "ETL / ELT": 5, "Agile / Scrum": 5,
    "Data warehouse": 5, "SAP": 5, "Statistics": 5, "LLM / GenAI": 5,
    "AWS": 5, "Excel": 5, "Git": 3, "Databricks": 5, "R": 5, "SAS": 5,
    "Spark": 5, "DAX / Power Query": 5, "Scala": 5, "C#": 5,
}

# Results of the earlier pattern versions (version 1, version 2)
spot_check_before = {"Agile / Scrum": [1, 2], "R": [4, 4]}

check = pd.DataFrame({"correct of 5": spot_check}).sort_values("correct of 5")
print(check.to_string())
print("\n", check["correct of 5"].sum(), "of", 5 * len(check), "checked hits correct")
print(len(check), "skills checked,", len(SKILLS) - len(check), "not checked\n")

print("earlier pattern versions (version 1, version 2):")
for skill, results in spot_check_before.items():
    print(f"  {skill}: {results} -> final {spot_check[skill]}")

### Checking the matches

**I read five random hits per skill in their context and counted how many really mean the skill.** A hit counts as correct when the word refers to a tool or method the job uses or requires (in the tasks, the requirements or the team's tech stack), including fields of study such as "Studium der Statistik". It counts as wrong when the word means something else: a company name, an application instruction, or a description of the employer.

The check covers 22 skills: the 15 most frequent ones and all skills with risky patterns (short names such as R, SAS or Git, and words with other meanings). **108 of 110 checked hits are correct.** The only skill below 5 of 5 is Git (3 of 5), where two hits are GitHub. The Git count (114 ads) is therefore likely too high: some ads mention GitHub only as a place for application documents.

**Two patterns were revised during the check:**

| Skill | Version 1 | Version 2 | Final | What changed |
|---|---|---|---|---|
| Agile / Scrum | 1 of 5 | 2 of 5 | 5 of 5 | first any "agil…" word, then "agil…" + a work word (which still matched "agile Arbeitskultur"); now only methods: Scrum, Kanban, "agile Methoden/Projekte" |
| R | 4 of 5 | 4 of 5 | 5 of 5 | "R+V" (an insurer), then "i. d. R." (*in der Regel*), are now excluded |

The 24 skills with unambiguous names (e.g. TensorFlow, Snowflake, Kafka) were not checked.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Muted, corporate style: desaturated slate blue, white background, soft dark-grey text
SURFACE, INK, INK_2, BAR = "#ffffff", "#2b2b2b", "#6b6b6b", "#5b7a99"
plt.rcParams.update({"font.family": "sans-serif",
                     "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"]})

top = counts.head(20)[::-1]                    # reversed so the most frequent skill is at the top
share = top / len(jobs) * 100
first, second = counts.index[:2]               # title is built from the data, not typed by hand

fig, ax = plt.subplots(figsize=(9, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

colors = [BAR if skill in (first, second) else "#c9ced6" for skill in top.index]
ax.barh(top.index, share, height=0.62, color=colors)

# Value at the tip of each bar (replaces the x-axis, so the axis is hidden)
for i, v in enumerate(share):
    ax.text(v + 0.6, i, f"{v:.0f}%", va="center", fontsize=10, color=INK_2)

# Title states the finding; subtitle says exactly what is measured
fig.text(0.02, 0.965, f"{first} and {second} are the most requested skills in German data jobs",
         fontsize=15, fontweight="bold", color=INK, ha="left")
fig.text(0.02, 0.93, f"Share of {len(jobs):,} data-role ads that mention each skill (top 20 of {len(SKILLS)})",
         fontsize=11, color=INK_2, ha="left")

# Source and method note
fig.text(0.02, 0.015,
         "Source: job search of the Bundesagentur für Arbeit, collected 7 October 2026. "
         "A skill counts when it is mentioned anywhere in the ad, including as nice to have.",
         fontsize=8.5, color=INK_2, ha="left")

# Recessive frame: no spines, no x-axis, no ticks
for side in ["top", "right", "bottom", "left"]:
    ax.spines[side].set_visible(False)
ax.xaxis.set_visible(False)
ax.tick_params(axis="y", length=0, labelsize=10.5, labelcolor=INK)
ax.set_xlim(0, share.max() * 1.12)             # room for the value labels

plt.subplots_adjust(left=0.2, right=0.97, top=0.9, bottom=0.06)
fig.savefig(FIG_DIR / "top_skills.png", dpi=200, facecolor=SURFACE)
plt.show()

### First result: the most frequent skills

**Python and SQL stand out: they are mentioned in 46% and 44% of the 1,062 data-role ads.** No other skill reaches a third of the ads. The next group consists of machine learning (30%), data modelling (30%), Power BI, Azure and ETL / ELT (each about 20%).

Two points stand out for the German market:

- **SAP appears in 18% of the ads.** A large part of data work in Germany happens inside SAP environments, which is easy to overlook when looking at international, English-language studies.
- **LLM / GenAI already appears in 14% of the ads**, about as often as AWS (13%).

The chart counts *mentions*: a skill counts as soon as it appears anywhere in the ad, also as "nice to have". The split into required and nice-to-have skills follows in the next step.

First the current state is saved to a file, so the next session can start without rerunning everything:

In [ ]:
# Checkpoint: roles, seniority and skills extracted (required vs. nice-to-have and barriers follow)
jobs_out = jobs.join(S.add_prefix("skill_"))   # one True/False column per skill, e.g. skill_Python
out = PROCESSED_DIR / "jobs_extracted.pkl"
jobs_out.to_pickle(out)
print(jobs_out.shape, "->", out.name)